In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pathlib import Path

In [2]:
spark = SparkSession.builder \
    .appName("create_custom_datasets") \
    .getOrCreate()

In [ ]:
# Path(__file__).resolve works only in .py file : file path
# In a notebook file, we use Path.cwd() : cwd
PROJECT_DIR = Path.cwd().parent
data_path  = PROJECT_DIR/"data"/"elec2_real_time_data.csv"
data_path = data_path.as_posix()
print(data_path)

c:/Users/ASUS/Desktop/Projects_and_research/kafka_pyspark/Projects/concept_drift/data/elec2_real_time_data.csv


In [24]:
df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(data_path)

In [4]:
df.show(10)

+--------+---+--------+--------+---------+--------+---------+--------+-----+
|    date|day|  period|nswprice|nswdemand|vicprice|vicdemand|transfer|class|
+--------+---+--------+--------+---------+--------+---------+--------+-----+
|0.884828|  3|0.170213|0.028612| 0.174204|0.001845|  0.27913|0.769298| DOWN|
|0.884828|  3|0.191489|0.027741|  0.18194|0.001784| 0.274728|     0.8| DOWN|
|0.884828|  3|0.212766|0.028672| 0.203065|0.001824|  0.28405|0.804825| DOWN|
|0.884828|  3|0.234043|0.030323|  0.24695|0.001942| 0.310461|0.787281|   UP|
|0.884828|  3|0.255319|0.040681|  0.33978|0.002636| 0.374158|0.663596|   UP|
|0.884828|  3|0.276596|0.043383| 0.437816| 0.00273|  0.43941|0.611404|   UP|
|0.884828|  3|0.297872|0.034676| 0.470842|0.002308| 0.485241|0.484649|   UP|
|0.884828|  3|0.319149|0.037378|  0.54582|0.002532| 0.558519|0.436404|   UP|
|0.884828|  3|0.340426|0.034526| 0.568581|0.002385| 0.589591|0.424561|   UP|
|0.884828|  3|0.361702|0.033866| 0.576465|0.002302| 0.596323|0.436404|   UP|

In [5]:
df.printSchema()

root
 |-- date: double (nullable = true)
 |-- day: integer (nullable = true)
 |-- period: double (nullable = true)
 |-- nswprice: double (nullable = true)
 |-- nswdemand: double (nullable = true)
 |-- vicprice: double (nullable = true)
 |-- vicdemand: double (nullable = true)
 |-- transfer: double (nullable = true)
 |-- class: string (nullable = true)



In [6]:
df = df.withColumn(
    "date", F.regexp_replace(F.col("date"), r"\.", "").cast(StringType())
).withColumn(
    "period", F.regexp_replace(F.col("period"), r"\.", "").cast(StringType())
).withColumn(
    "day", F.col("day").cast(StringType())
)

df = df.withColumn(
    "eventID", F.concat(F.col("date"), F.col("day"), F.col("period"))
)
df.show(10)

+-------+---+-------+--------+---------+--------+---------+--------+-----+---------------+
|   date|day| period|nswprice|nswdemand|vicprice|vicdemand|transfer|class|        eventID|
+-------+---+-------+--------+---------+--------+---------+--------+-----+---------------+
|0884828|  3|0170213|0.028612| 0.174204|0.001845|  0.27913|0.769298| DOWN|088482830170213|
|0884828|  3|0191489|0.027741|  0.18194|0.001784| 0.274728|     0.8| DOWN|088482830191489|
|0884828|  3|0212766|0.028672| 0.203065|0.001824|  0.28405|0.804825| DOWN|088482830212766|
|0884828|  3|0234043|0.030323|  0.24695|0.001942| 0.310461|0.787281|   UP|088482830234043|
|0884828|  3|0255319|0.040681|  0.33978|0.002636| 0.374158|0.663596|   UP|088482830255319|
|0884828|  3|0276596|0.043383| 0.437816| 0.00273|  0.43941|0.611404|   UP|088482830276596|
|0884828|  3|0297872|0.034676| 0.470842|0.002308| 0.485241|0.484649|   UP|088482830297872|
|0884828|  3|0319149|0.037378|  0.54582|0.002532| 0.558519|0.436404|   UP|088482830319149|

In [7]:
df_features = df.select(
    "eventID",
    "nswprice",
    "nswdemand",
    "vicprice",
    "vicdemand",
    "transfer"
)
df_features.show(10)

+---------------+--------+---------+--------+---------+--------+
|        eventID|nswprice|nswdemand|vicprice|vicdemand|transfer|
+---------------+--------+---------+--------+---------+--------+
|088482830170213|0.028612| 0.174204|0.001845|  0.27913|0.769298|
|088482830191489|0.027741|  0.18194|0.001784| 0.274728|     0.8|
|088482830212766|0.028672| 0.203065|0.001824|  0.28405|0.804825|
|088482830234043|0.030323|  0.24695|0.001942| 0.310461|0.787281|
|088482830255319|0.040681|  0.33978|0.002636| 0.374158|0.663596|
|088482830276596|0.043383| 0.437816| 0.00273|  0.43941|0.611404|
|088482830297872|0.034676| 0.470842|0.002308| 0.485241|0.484649|
|088482830319149|0.037378|  0.54582|0.002532| 0.558519|0.436404|
|088482830340426|0.034526| 0.568581|0.002385| 0.589591|0.424561|
|088482830361702|0.033866| 0.576465|0.002302| 0.596323|0.436404|
+---------------+--------+---------+--------+---------+--------+
only showing top 10 rows



In [8]:
df_features.printSchema()

root
 |-- eventID: string (nullable = true)
 |-- nswprice: double (nullable = true)
 |-- nswdemand: double (nullable = true)
 |-- vicprice: double (nullable = true)
 |-- vicdemand: double (nullable = true)
 |-- transfer: double (nullable = true)



In [9]:
pdf = df_features.toPandas()
pdf.to_csv("../data/elec2_real_time_features.csv", index=False)

In [10]:
df_labels = df.select(
    F.col("eventID"),
    F.col("class").alias("label")
)
df_labels.show(10)

+---------------+-----+
|        eventID|label|
+---------------+-----+
|088482830170213| DOWN|
|088482830191489| DOWN|
|088482830212766| DOWN|
|088482830234043|   UP|
|088482830255319|   UP|
|088482830276596|   UP|
|088482830297872|   UP|
|088482830319149|   UP|
|088482830340426|   UP|
|088482830361702|   UP|
+---------------+-----+
only showing top 10 rows



In [11]:
df_labels.printSchema()

root
 |-- eventID: string (nullable = true)
 |-- label: string (nullable = true)



In [12]:
pdf = df_labels.toPandas()
pdf.to_csv("../data/elec2_true_labels.csv", index=False)

In [25]:
spark.stop()